In [ ]:
import json
import os, getpass
from langchain.chat_models import init_chat_model
from IPython.display import Image, display


def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")


_set_env("OPENAI_API_KEY")
_set_env("LANGCHAIN_API_KEY")
_set_env("MISTRAL_API_KEY")
_set_env("TOGETHER_API_KEY")

In [2]:
from langchain_openai import ChatOpenAI


llm = ChatOpenAI(model="gpt-4o-2024-11-20", temperature=0.7)

In [18]:
import json
import pandas as pd
def _get_user_intents():
    # Read JSON from a file
    with open('prompt/question_intent.json', 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Now `data` is a Python dictionary or list
    return data


def _get_nasa_taxonomy():
    # Read JSON from a file
    with open('../gcmd_science_keywords/independent_vertices.json', 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Now `data` is a Python dictionary or list
    return pd.DataFrame(data)



def _get_random_terms(n=3):
    nasa  = _get_nasa_taxonomy()

    # only topics
    topic_df = nasa[nasa["level"] == "Topic"]

    # get terms
    topics_uuid = topic_df["uuid"].unique().tolist()
    term_df = nasa[nasa["parent_uuid"].isin(topics_uuid)]

    topics= []
    for topic, term_df in term_df.groupby("parent_uuid"):  # group by topic
        # get topic info
        topic_info = nasa[nasa["uuid"] == topic]
        topic_name = topic_info["name"].values[0]
        topic_description = topic_info["description"].values[0]

        # get 4 random topics

        term_sample_df = term_df.sample(n=n, random_state=42)
        context_str = f"**TOPIC**: {topic_name} - {topic_description}. \n\n"
        terms = term_sample_df["name"].values.tolist()
        
        for _,term_info in term_sample_df.iterrows():
            # get term info
            term_name = term_info["name"]
            term_description = term_info["description"]

            context_str = f"{context_str}- *sSUBJECT AREA*: {term_name}\n{term_description}\n"

        context_str = f"{context_str}"

        topics.append({
            "topic": topic_name,
            "terms": terms,
            "context": context_str
        })
    return topics



In [19]:
from langchain_core.messages import HumanMessage, SystemMessage
from tqdm import tqdm


def generate_prompts():
    questions = []

    topic_contexts = _get_random_terms()
    intents = _get_user_intents()

    with tqdm(total=len(topic_contexts) * len(intents)) as pbar:
        for topic_contexts in topic_contexts: 
            for item in intents:
                pbar.update(1)
                intent_category = item["intent"]
                intent_description = item["description"]
                
                with open("prompt/system.txt", "r") as file:
                    system = file.read()

                with open("prompt/generate_question.txt", "r") as file:
                    instructions = file.read()

                instructions = instructions.format(intent_category= intent_category, intent_description= intent_description, context = topic_contexts["context"])
                
                question = llm.invoke(
                    [
                        SystemMessage(system),
                        HumanMessage(instructions),
                    ],
                ).content
                
                q = {
                    "idx" : f'{topic_contexts["topic"]}_{intent_category}',
                    "intent": intent_category,
                    **topic_contexts,
                    "instructions": instructions,
                    "question": question
                    }
                questions.append(q)
            
    return pd.DataFrame(questions)

In [ ]:
questions  = generate_prompts()

  0%|          | 0/70 [00:00<?, ?it/s]

 79%|███████▊  | 55/70 [01:05<00:25,  1.73s/it]